# BaCuSe Bayesian optimization from XRD metrics

The objective and Gaussian-process suggestion functions live in `xrd_bo_optimization_tf.bayesian_opt`. Generated suggestions and plots are written under `data/results/` and `outputs/figures/`.

In [ ]:
import pandas as pd

from xrd_bo_optimization_tf.bayesian_opt import prepare_optimization_data
from xrd_bo_optimization_tf.config import (
    BO_PARAMETER_COLUMNS,
    GENERATED_XRD_METRICS_CSV,
    XRD_METRICS_CSV,
)

metrics_path = (
    GENERATED_XRD_METRICS_CSV
    if GENERATED_XRD_METRICS_CSV.is_file()
    else XRD_METRICS_CSV
)
metrics = pd.read_csv(metrics_path)
df_bo = prepare_optimization_data(metrics)
print(f'Loaded {len(metrics)} metric rows from {metrics_path}')
print(f'Using {len(df_bo)} complete observations for optimization')

In [ ]:
columns = ['sample_id', *BO_PARAMETER_COLUMNS, 'crystallinity', 'phase_purity', 'crystallinity_norm', 'objective']
print(df_bo[columns].sort_values('objective', ascending=False).round(3).to_string(index=False))
best = df_bo.loc[df_bo['objective'].idxmax()]
print(f'\nBest observed objective: {best["objective"]:.3f} ({best["sample_id"]})')

In [ ]:
import matplotlib.pyplot as plt

from xrd_bo_optimization_tf.bayesian_opt import save_suggestions, suggest_next_conditions
from xrd_bo_optimization_tf.config import FIGURES_DIR
from xrd_bo_optimization_tf.plotting import plot_bo_suggestions, plot_parameter_metrics

suggestions = suggest_next_conditions(df_bo, n_suggestions=5, random_state=0)
suggestions_path = save_suggestions(suggestions)
print(suggestions.round(3).to_string(index=False))
print(f'\nSaved suggestions to {suggestions_path}')

objective_figure = plot_bo_suggestions(
    df_bo, suggestions, FIGURES_DIR / 'bacuse2_bo_suggestions.png'
)
parameter_figure = plot_parameter_metrics(
    df_bo, suggestions, output_path=FIGURES_DIR / 'bacuse2_bo_parameter_plots.png'
)
plt.show()